In [65]:
"""
Copyright (C) <2025>  <The Ohio State University>

This program is free software: you can redistribute it and/or modify it under
the terms of the GNU General Public License as published by the Free Software
Foundation, either version 3 of the License, or (at your option) any later version.
This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
See the GNU General Public License for more details. You should have received a copy of the
GNU General Public License along with this program.  If not, see <https://www.gnu.org/licenses/>

This script processes and analyzes tidied data of ∆∆G and Kd ratios for human
transcripts with indels. It reads a file containing ∆∆G values for various
indel sizes at different positions relative to a binding site. It calculates
the mean and standard deviation of ∆∆G and Kd ratios, grouping the data
by indel size (1, 2, 3, 4, and 5-10 nucleotides) and position.
""";


In [66]:
%reset -f
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [67]:
import numpy as np
import math
from scipy.stats import bootstrap
import os

In [68]:
working_directory = "/content/drive/MyDrive/work/data/RNA-protein/IndelPolymorphismsModulateDistalRNAProteinInteractions"
os.chdir(working_directory)

In [69]:
# --- Script Parameters ---
seq_length = 150
footprint = 7

In [70]:
file_names = ['dataHuman/sameSites/all/sameSites150',
              'dataHuman/sameSites/restricted/sameSites150',
              'dataHuman/badSites/all/badSites150',
              'dataHuman/badSites/restricted/badSites150']

# the dataset currently being processed
file_name = file_names[3]

# Output files for standard deviation, confidence intervals, variance, and mean
infilename = "/".join(file_name.split("/")[:-1]) + "/tidiedKdDDG.txt"
infile = open(infilename, 'r')

dev_outfilename = file_name +'_dev.dat'
dev_outfile = open(dev_outfilename, 'w')

ci_outfilename = file_name + '_dev_ci.dat'
ci_outfile = open(ci_outfilename, 'w')

var_outfilename = file_name + '_var.dat'
var_outfile = open(var_outfilename, 'w')

avg_outfilename = file_name + '_avg.dat'
avg_outfile = open(avg_outfilename, 'w')

stats_outfilename = file_name + '_stats.dat'
stats_outfile = open(stats_outfilename, 'w')

In [71]:
# --- Helper function to calculate bootstrap CI for standard deviation ---
def get_stdev_ci(data):
    """
    Calculates the 95% bootstrap confidence interval for the standard deviation.

    Args:
        data (list or np.array): A list of numerical data.

    Returns:
        tuple: A tuple containing the lower and upper bounds of the confidence
               interval.
    """

    # Ensure data is a numpy array for the bootstrap function
    data_arr = np.array(data)

    # The bootstrap function requires samples to be passed as a tuple, so we pass (data_arr,).
    res = bootstrap((data_arr,),
                    np.std,
                    confidence_level=0.95,
                    random_state=1,
                    n_resamples=5000)#batch=batch_size)

    return (res.confidence_interval.low, res.confidence_interval.high)

In [72]:
# --- Data Structures ---
# Initialize nested lists to store DDG values.

DDGs = [[[] for i in range(45)] for i in range(5)]

avg_DDG = [[] for i in range(5)]
dev_DDG = [[] for i in range(5)]
dev_ci_DDG = [[] for i in range(5)]
var_DDG = [[] for i in range(5)]

positions = [i-1 for i in range(45)]

In [73]:
# Define a tab character for formatting the output strings.
t='\t'

# --- Data Loading ---
# Skip the header line of the input file.
infile.readline()

# Read the data file line by line and populate the data structures.
for line in infile:
    fields = line.split()
    position = int(fields[0])
    DDG = fields[2]
    deletionSize = int(fields[3])

    # Skip positions outside the defined range.
    if position > 43:
        continue

    # Group data by deletion size. Sizes 1-4 get their own category.
    # Sizes >= 5 are grouped into a single category (index 4).
    if deletionSize < 5:
      DDGs[deletionSize-1][position+1].append(float(DDG))
    else:
      DDGs[4][position+1].append(float(DDG))

In [74]:
# --- Write Data Statistics ---
# Write a file showing the number of data points for each category.
statsHeader ="\t".join(["deletion_size"] + [str(i-1) for i in range(45)]+["\n"])

stats_outfile.write(statsHeader)
for i in range(5):
  deletions = ["1del", "2dels", "3dels", "4dels", "5to10"]
  stats_outfile.write(deletions[i]+"\t\t")
  for j in range(45):
    stats_outfile.write(str(len(DDGs[i][j])) + t)
  stats_outfile.write("\n")
stats_outfile.close()

In [75]:
# --- Subsample large datasets for consistent statistics ---
subsample_size = 20000


for i in range(len(DDGs)):
    for j in range(len(DDGs[i])):
        if len(DDGs[i][j]) > subsample_size:
            np.random.seed(1)
            DDGs[i][j] = np.random.choice(DDGs[i][j], size=subsample_size, replace=False)

In [76]:
# --- Calculate and Store Statistics ---
# Iterate through each indel size and position to calculate statistics.
for i in range(5):
    for j in range(45):
        # Calculate mean and standard deviation for DDG and KD.
        avg_DDG[i].append(np.mean(DDGs[i][j]))
        var_DDG[i].append(np.var(DDGs[i][j]))
        dev_DDG[i].append(math.sqrt(np.var(DDGs[i][j])))

        # Calculate confidence interval for the standard deviation of DDG.
        low, high = get_stdev_ci(DDGs[i][j])
        dev_ci_DDG[i].append((low, high))

In [77]:
# Define and write headers for the output files.
header = "# \t position \t 1deletion \t 2deletion \t 3deletion \t 4deletion \t 5to10deletions \n"
header_ci = "# \t position \t 1del_low \t 1del_high \t 2del_low \t 2del_high \t 3del_low \t 3del_high \t 4del_low \t 4del_high \t 5to10del_low \t 5to10del_high \n"

dev_outfile.write(header)
var_outfile.write(header)
avg_outfile.write(header)
ci_outfile.write(header_ci)


# Iterate through positions and write the calculated statistics to files.
for i in positions:
    # Format strings for each output file.
    string_avg = str(i) + t + str(avg_DDG[0][i]) + t + str(avg_DDG[1][i]) + t + str(avg_DDG[2][i]) + t + str(avg_DDG[3][i]) + t + str(avg_DDG[4][i])
    string_dev = str(i) + t + str(dev_DDG[0][i]) + t + str(dev_DDG[1][i]) + t + str(dev_DDG[2][i]) + t + str(dev_DDG[3][i]) + t + str(dev_DDG[4][i])
    string_var = str(i) + t + str(var_DDG[0][i]) + t + str(var_DDG[1][i]) + t + str(var_DDG[2][i]) + t + str(var_DDG[3][i]) + t + str(var_DDG[4][i])

    # Create string for CI data (low and high values for each deletion size).
    ci_vals = [f"{dev_ci_DDG[k][i][0]}\t{dev_ci_DDG[k][i][1]}" for k in range(5)]
    string_ci = str(i) + "\t" + "\t".join(ci_vals)

    # Write strings to the corresponding files.
    dev_outfile.write(string_dev + "\n")
    var_outfile.write(string_var + "\n")
    avg_outfile.write(string_avg + "\n")
    ci_outfile.write(string_ci + "\n")

In [78]:
# Close all open file handles.
infile.close()
dev_outfile.close()
var_outfile.close()
avg_outfile.close()
ci_outfile.close()